In [1]:
# TO-DO: test opening obs_seq.out as pandas df and then concat then reshape and output xr.dataset

In [2]:
import pydartdiags.obs_sequence.obs_sequence as obsq
import xarray as xr
import sys
import os
import dask
import numpy as np
import pandas as pd
import true_w_mean as truew
from pathlib import Path

module_dir = Path("/glade/work/iranjan/tpose24-osse/")
sys.path.append(str(module_dir))

import osse_tools as ost

In [3]:
path = "/glade/derecho/scratch/iranjan/eep-osse-30/outputs/"
files = "obs_file-*.nc"
input_path = os.path.join(path, files)

In [4]:
def join_obs_seq_outputs(output_dir: str) -> obsq.ObsSequence:
    """
    Reads all obs_seq_*.out files from a directory and joins them into
    a single ObsSequence using obsq.ObsSequence.join().

    Args:
        output_dir (str): Directory containing obs_seq_*.out files.

    Returns:
        obsq.ObsSequence: Single joined ObsSequence containing all
            observations from all files, sorted by time.
    """
    output_path = Path(output_dir)
    out_files = sorted(output_path.glob("*_obs_seq.out"))

    if not out_files:
        raise FileNotFoundError(f"No *_obs_seq.out files found in {output_dir}")

    print(f"Found {len(out_files)} obs_seq.out files")

    obs_seqs = []
    for f in out_files:
        print(f"  reading: {f.name}")
        obs_seqs.append(obsq.ObsSequence(str(f)))

    print("Joining...")
    joined = obsq.ObsSequence.join(obs_seqs)
    print(f"Total obs in joined sequence: {len(joined.df)}")

    return joined


In [5]:
def _infer_cadence(times):
    """
    Infer the nominal sampling cadence from the data itself, as the most
    common gap between consecutive unique timestamps. This avoids hardcoding
    a cadence that might not match your actual obs interval.
    """
    unique_sorted = np.sort(pd.unique(times))
    if len(unique_sorted) < 2:
        return pd.Timedelta(0)
    diffs = np.diff(unique_sorted)
    vals, counts = np.unique(diffs, return_counts=True)
    return pd.Timedelta(vals[np.argmax(counts)])

In [6]:
def reshape_obs_to_uv_samples(df, depth_round=None, time_tolerance=None):
    """
    Convert a flat obs DataFrame (as returned by load_obs_files_to_df) into
    the gridded (time, glider, obs_depth) structure compute_w_planefit
    requires.
 
    Parameters
    ----------
    df : pd.DataFrame
        Concatenated obs rows with 'type', 'observation', 'latitude',
        'longitude', 'vertical', 'time' columns. 'vertical' assumed
        POSITIVE-down and will be negated.
    depth_round : int or None
        Decimal places to round 'vertical' to, if depths carry float noise.
    time_tolerance : pd.Timedelta or None
        Snap timestamps within this tolerance of the inferred nominal
        cadence's grid to a shared value, so gliders sampling at "the same"
        nominal time but with slightly different clock offsets get grouped
        into one (time, glider, obs_depth) cell instead of each getting
        their own near-duplicate time coordinate (which is what was
        blowing up the pivot). If None, tolerance defaults to half the
        inferred cadence.
    """
    df = df.copy()
 
    # --- 1. snap timestamps to a shared nominal grid ---
    cadence = _infer_cadence(df['time'])
    if cadence <= pd.Timedelta(0):
        print("WARNING: could not infer a nonzero cadence from timestamps; "
              "skipping time snapping. Check df['time'] manually.")
    else:
        tol = time_tolerance if time_tolerance is not None else cadence / 2
        t0 = df['time'].min()
        offset = (df['time'] - t0)
        n_steps = (offset / cadence).round()
        snapped = t0 + n_steps * cadence
        max_shift = (df['time'] - snapped).abs().max()
        if max_shift > tol:
            print(f"WARNING: snapping timestamps to inferred cadence "
                  f"({cadence}) required shifts up to {max_shift}, which "
                  f"exceeds tolerance ({tol}). Cadence may be mis-detected "
                  f"— inspect df['time'].diff() manually before trusting "
                  f"this pivot.")
        df['time'] = snapped
        print(f"Inferred cadence: {cadence}. Snapped timestamps to this grid "
              f"(max shift applied: {max_shift}).")
 
    # --- 2. assign glider id from unique (lat, lon) pairs ---
    pos_key = list(zip(df['latitude'].round(6), df['longitude'].round(6)))
    unique_pos = sorted(set(pos_key))
    pos_to_glider = {p: i for i, p in enumerate(unique_pos)}
    df['glider'] = [pos_to_glider[p] for p in pos_key]
    print(f"Found {len(unique_pos)} distinct glider positions.")
 
    # --- 3. flip depth sign: positive-down -> negative-down ---
    depth = -df['vertical'].astype(float)
    if depth_round is not None:
        depth = depth.round(depth_round)
    df['obs_depth'] = depth

    # --- 4. split U and V by type ---
    is_u = df['type'].str.contains('U_CURRENT')
    is_v = df['type'].str.contains('V_CURRENT')
    if not is_u.any() or not is_v.any():
        raise ValueError(
            f"Expected types containing 'U_CURRENT' / 'V_CURRENT', "
            f"found: {df['type'].unique()}"
        )
    df_u = df.loc[is_u, ['time', 'glider', 'obs_depth', 'observation', 'latitude', 'longitude']]
    df_v = df.loc[is_v, ['time', 'glider', 'obs_depth', 'observation', 'latitude', 'longitude']]
 
    # --- 5. check for duplicate (time, glider, obs_depth) after snapping ---
    for name, d in [('U', df_u), ('V', df_v)]:
        dup = d.duplicated(subset=['time', 'glider', 'obs_depth'], keep=False)
        if dup.any():
            print(f"WARNING: {dup.sum()} duplicate (time, glider, obs_depth) rows "
                  f"in {name} after time snapping — pivot will keep only the "
                  f"last of each. If this count is large, your inferred "
                  f"cadence/tolerance may be too coarse.")
 
    # --- 6. pivot each to (time, glider, obs_depth) ---
    u_da = (df_u.set_index(['time', 'glider', 'obs_depth'])['observation']
                .to_xarray().rename('U'))
    v_da = (df_v.set_index(['time', 'glider', 'obs_depth'])['observation']
                .to_xarray().rename('V'))
 
    # --- 7. align U and V ---
    u_da, v_da = xr.align(u_da, v_da, join='outer')
    n_nan_u = int(u_da.isnull().sum())
    n_nan_v = int(v_da.isnull().sum())
    if n_nan_u or n_nan_v:
        print(f"WARNING: after aligning U and V, found {n_nan_u} NaN in U and "
              f"{n_nan_v} NaN in V. compute_w_planefit's plane fit does not "
              f"handle NaNs — drop or fill before running it.")
 
    # --- 8. one lat/lon per glider (mean position) ---
    glider_lat = df.groupby('glider')['latitude'].mean()
    glider_lon = df.groupby('glider')['longitude'].mean()
    pos_std = df.groupby('glider')[['latitude', 'longitude']].std().max().max()
    if pos_std > 1e-6:
        print(f"NOTE: glider positions vary within a glider (max std "
              f"{pos_std:.4g} deg) — using the mean position collapses that "
              f"drift into a single point for the plane fit.")
 
    out = xr.Dataset({'U': u_da, 'V': v_da})
    out = out.assign_coords(
        lat=('glider', glider_lat.reindex(out.glider.values).values),
        lon=('glider', glider_lon.reindex(out.glider.values).values),
    )
    out = out.sortby('obs_depth', ascending=False)
    # --- 9. confirm uniform depth spacing across the full range ---
    depths = np.sort(out.obs_depth.values)
    dz = np.diff(depths)
    if not np.allclose(dz, dz[0], rtol=1e-4):
        print(f"WARNING: obs_depth spacing is not uniform across the full range "
              f"(min dz={dz.min()}, max dz={dz.max()}).")
 
    return out

In [7]:
joined_obs_seq = join_obs_seq_outputs(path)

Found 248 obs_seq.out files
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-001-03_obs_seq.out
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-001-06_obs_seq.out
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-001-09_obs_seq.out
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-001-12_obs_seq.out
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-001-15_obs_seq.out
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-001-18_obs_seq.out
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-001-21_obs_seq.out
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-001-24_obs_seq.out
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-002-03_obs_seq.out
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-002-06_obs_seq.out
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-002-09_obs_seq.out
  reading: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-002-12_

In [8]:
uv_samples = reshape_obs_to_uv_samples(joined_obs_seq.df)

Inferred cadence: 0 days 03:00:00. Snapped timestamps to this grid (max shift applied: 0 days 00:00:00).
Found 4 distinct glider positions.


In [9]:
est_w = ost.compute_w_planefit(uv_samples, extrapolate_to_surface=False)

In [10]:
print(est_w)

<xarray.Dataset> Size: 151kB
Dimensions:    (time: 248, depth: 38, obs_depth: 37)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2015-01-01T01:00:00 ... 2015-01-31T2...
  * depth      (depth) float64 304B -7.0 -9.0 -11.0 -13.0 ... -77.0 -79.0 -81.0
  * obs_depth  (obs_depth) float64 296B -8.0 -10.0 -12.0 ... -76.0 -78.0 -80.0
Data variables:
    w_est      (time, depth) float64 75kB 0.0 2.004e-07 ... 2.45e-05 2.292e-05
    div        (time, obs_depth) float64 73kB 1.002e-07 ... -7.875e-07


In [11]:
print(true_w)

NameError: name 'true_w' is not defined